### Part 1: Setup

In [1]:
# Install Unsloth (handles LoRA, quantization, fast training)
# !pip install -q unsloth
# !pip install -q datasets evaluate rouge_score

# Supress noisy deprication warnings
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", message=".*use_return_dict.")
warnings.filterwarnings("ignore", message=".*max_new_tokens.*max_length.*")


In [3]:
import torch, json, random, math, re, os
from unsloth import FastLanguageModel
from pathlib import Path

print(f"PyTorch:  {torch.__version__}")
print(f"GPU:      {torch.cuda.get_device_name(0)}")
print(f"VRAM:     {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch:  2.11.0+cu130
GPU:      NVIDIA GeForce RTX 3070
VRAM:     8.6 GB


### Load Base Model

In [38]:
BASE_MODEL = "HuggingFaceTB/SmolLM-135M"
MAX_SEQ_LENGTH = 512
SEED = 42

# Load the base model (no fine-tuning yet)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

# Set up for inference
FastLanguageModel.for_inference(model)

print(f"\n[OK] Model loaded: {BASE_MODEL}")
print(f"Parameters: {sum(p.numel() for p in model.parameters())}")

==((====))==  Unsloth 2025.11.1: Fast Llama patching. Transformers: 4.57.2.
   \\   /|    NVIDIA GeForce RTX 3070. Num GPUs = 1. Max memory: 8.0 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 8.6. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!

[OK] Model loaded: HuggingFaceTB/SmolLM-135M
Parameters: 81430848


In [27]:
# Helper function to generate text
def generate_text(model, tokenizer, prompt, max_new_tokens=100):
    """Generate a completion for a given prompt"""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Ensure model is on the correct device
    model = model.to(device)

    # Tokenize input and move to the same device as the model
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LENGTH
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            inputs=inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_new_tokens=max_new_tokens,
            use_cache=True,
            do_sample=True,
            repetition_penalty=1.2,
            temperature=0.7,
            top_p=0.9,
        )

    # Only decode the new tokens (not the prompt)
    new_tokens = outputs[0, inputs.input_ids.shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

In [6]:
# Test the base model on financial prompts
financial_prompts = [
  "The company reported total revenue of",
  "Risk factors include exposure to fluctuations in",
  "Diluted earnings per share for the fiscal year ended",
  "The Board of Directors declared a quarterly dividend of",
  "Our long-term debt obligations consist primarily of",
]

print("=" * 60)
print("BASE MODEL -- Financial Text Generation")
print("=" * 60)
for prompt in financial_prompts:
  completion = generate_text(model, tokenizer, prompt, max_new_tokens=60)
  print("-" * 60)
  print(f"\nPrompt: {prompt}")
  print(f"Completion: {completion}")
  print("-" * 60)

BASE MODEL -- Financial Text Generation
------------------------------------------------------------

Prompt: The company reported total revenue of
Completion:  $4,502 million in 1986 and a decrease at the end.
Minnesota’s income statement shows its earnings per share (EPS) as follows:
|Year||Net Income||$37,725,285|
------------------------------------------------------------
------------------------------------------------------------

Prompt: Risk factors include exposure to fluctuations in
Completion:  income, family structure (e.g., grandparents) and/or living conditions such as poverty or homelessness,” says, “and it is possible for children who have experienced adversity before them at some point.”
“Children with disabilities are particularly vulnerable because of the stigma associated with disability that can be
------------------------------------------------------------
------------------------------------------------------------

Prompt: Diluted earnings per share for the fi

Observation: The base model generates generic text. It has no idea what financial language sounds like. It produces grammatical English, but i won't sound like a 10-K filing.

In [ ]:
# measures how surprised model is by comparing generated token with actual next token
def compute_preplixity(model, tokenizer, texts, max_length=512):
    """
    Preplexity = e^(avg cross-entropy loss)
    lower = model is less surprised by the text = better fit

    This is the metric for CPT. If preplexity drops on domain text
    after training, the model learned the domain.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Ensure model is on the correct device
    model = model.to(device)

    total_loss = 0
    total_tokens = 0

    model.eval()
    for text in texts:
        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=max_length
        ).to(device)  # Move inputs to the same device as the model

        with torch.no_grad():
            outputs = model(**inputs, labels=inputs.input_ids)

        num_tokens = inputs.input_ids.shape[1]
        total_loss += outputs.loss.item() * num_tokens
        total_tokens += num_tokens

    avg_loss = total_loss / total_tokens
    return math.exp(avg_loss)

In [8]:
# Quick preplexity check on sample financial text
sample_financial = [
  "The Company recognized impairment charges of $142 million related to goodwill associated with the North America reporting unit during the fiscal year ended March 31, 2024.",
  "Net cash provided by operating activities was $3.8 billion for the year ended December 31, 2023, compared to $4.1 billion for the year ended December 31, 2022.",
  "Total stockholders equity decreased to $12.4 billion as of September 30, 2023, primarily due to share repurchases of $2.1 billion and dividend payments of $800 million.",
]

base_ppl = compute_preplixity(model, tokenizer, sample_financial)

# Higher = more surprised = doesn't know domain well
print(f"Base Model preplexity on finiancial text: {base_ppl:.1f}")


Base Model preplexity on finiancial text: 11.4


### Prepare the Data

In [32]:
from datasets import load_dataset

print("[...] Loading SEC 10-K filings from HuggingFace...")
print("This is the PleIAs/SEC dataset -- full annual reports 1993-2024.")

# Stream to avoid downloadingn 100GB+
sec_dataset = load_dataset(
    "PleIAs/SEC",
    split="train",
    streaming=True,
)

# Collect a subset of filings
NUM_TRAIN = 240
NUM_VAL = 60
all_texts = []

print(f"[...] Collecting {NUM_TRAIN + NUM_VAL} filings...")
for i, example in enumerate(sec_dataset):
  text = example.get("text", "")
  # Filter: only keep substantial filings (>1000 words)
  if text and len(text.split()) > 1000:
    all_texts.append(text)
  if len(all_texts) >= NUM_TRAIN + NUM_VAL:
    break
  if i % 500 == 0 and i > 0:
    print(f"  Scanned {i} entries, collected {len(all_texts)} so far...")

print(f"\n[OK] Collected {len(all_texts)} filings.")

[...] Loading SEC 10-K filings from HuggingFace...
This is the PleIAs/SEC dataset -- full annual reports 1993-2024.
[...] Collecting 300 filings...

[OK] Collected 300 filings.


In [33]:
# Let's look at what a 10-K filing looks like
sample = all_texts[0]  # Just a random sample
words = sample.split()

print(f"Sample filing length: {len(words)} words\n")
print(f"First 200 words: { ' '.join(words[:200]) }")
print("...")
print(f"Last 200 words: { ' '.join(words[-200:]) }")

Sample filing length: 18357 words

First 200 words: ITEM 1. BUSINESS ~~~~~~~ ~~~~~~~~ OVERVIEW ~~~~~~~~ Cummins Engine Company, Inc. ("Cummins" or "the Company") is a leading worldwide designer and manufacturer of fuel-efficient diesel engines and related products. Engines ranging from 76 to 2,000 horsepower serve a wide variety of equipment in Cummins' key markets: heavy-duty truck, midrange truck, power generation, bus and light commercial vehicles, industrial products, government and marine. In addition, Cummins produces strategic components and subsystems critical to the engine, including filters, turbochargers and electronic control systems. Cummins sells its products to original equipment manufacturers ("OEMs"), distributors and other customers worldwide and conducts manufacturing, sales, distribution and service activities in most areas of the world. In 1993, approximately 56 percent of net sales were made in the United States. Major international markets include the United King

### Data Cleaning
remove boilerplate at end (exhibits, signatures, leagal text)

In [34]:
def clean_sec_filing(text):
  """
  Clean a SEC 10-K filing:
  1. Strip trailing exhibits/signatures/legal text
  2. Remove excessive whitespace
  3. Keep the substantive bussness/financial content
  """
  end_markers = [
      r"(?i)\bEXHIBIT\s+INDEX\b",
      r"(?i)\bSIGNATURES\b",
      r"(?i)\bPOWER\s+OF\s+ATTORNEY\b",
      r"(?i)\bEXHIBIT\s+\d",
  ]

  earliest_end = len(text)
  for pattern in end_markers:
    matches = list(re.finditer(pattern, text))
    if matches:
      last_match = matches[-1]
      if last_match.start() > 0.7 * len(text):
        earliest_end = min(earliest_end, last_match.start())

  text = text[:earliest_end]
  text = re.sub(r"\n{3,}", "\n\n", text)
  text = re.sub(r" {2,}", " ", text)
  text = text.strip()

  return text

# Clean all filings
clean_texts = []
for i, text in enumerate(all_texts):
  orignal_len = len(text.split())
  cleaned = clean_sec_filing(text)
  cleaned_len = len(cleaned.split())

  if cleaned_len > 500:
    clean_texts.append(cleaned)

  if i < 3:
    removed_pct = (1 - cleaned_len / orignal_len) * 100
    print(f"Filing {i}: {orignal_len} words -> {cleaned_len} words ({removed_pct:.1f}%)")

print(f"\n[OK] Cleaned {len(clean_texts)} filings.")

Filing 0: 18357 words -> 17521 words (4.6%)
Filing 1: 2727 words -> 2683 words (1.6%)
Filing 2: 6874 words -> 5451 words (20.7%)

[OK] Cleaned 300 filings.


### Chunking
divide sec filings to fit models context window

In [35]:
def chunk_texts(texts, chunk_size=256, overlap=0.2):
  """
  Split texts into overlapping chunks.

  Args:
    texts: List of strings to chunk
    chunk_size: Number of tokens per chunk
    overlap: Overlap between chunks as a fraction
  """
  step = int(chunk_size * (1 - overlap))
  all_chunks = []

  for text in texts:
    words = text.split()
    for i in range(0, len(words), step):
      chunk = words[i:i + chunk_size]
      if len(chunk) > 50: # Skip tiny trailing chunks
        all_chunks.append(" ".join(chunk))

  return all_chunks


# Split into train and val First (at filing level, not chunk level)
# This prevent data leakage
random.seed(SEED)
random.shuffle(clean_texts)

train_filings = clean_texts[:NUM_TRAIN]
val_filings = clean_texts[NUM_TRAIN:NUM_TRAIN + NUM_VAL]

print(f"Train filings: {len(train_filings)}")
print(f"Val filings: {len(val_filings)}")

# Now chunk each set
train_chunks = chunk_texts(train_filings)
val_chunks = chunk_texts(val_filings)

print(f"Train chunks: {len(train_chunks)}")
print(f"Val chunks: {len(val_chunks)}")

print(f"\nSample fhunk (first 100 words)")
print(" ".join(train_chunks[0].split()[:100]))

Train filings: 240
Val filings: 60
Train chunks: 19543
Val chunks: 5190

Sample fhunk (first 100 words)
Item 1. BUSINESS Ogden Corporation (hereafter together with its consolidated subsidiaries referred to as "Ogden" or the "Company") has its offices located at Two Pennsylvania Plaza, New York, New York 10121, pursuant to a lease that expires on April 30, 1998 and which contains an option by Ogden to renew for an additional five years. Ogden is a diversified company primarily engaged in providing services through subsidiaries within each of its Operating Services and Waste-to-Energy Operations as described below: I. OPERATING SERVICES Ogden Services Corporation ("Ogden Services"), a wholly owned subsidiary of Ogden, provides services through each of its operating


In [36]:
# Conver to HuggingFace Dataset format
from datasets import Dataset

train_dataset = Dataset.from_dict({"text": train_chunks})
val_dataset = Dataset.from_dict({"text": val_chunks})

print(f"Train dataset: {(train_dataset)}")
print(f"Val dataset: {(val_dataset)}")

Train dataset: Dataset({
    features: ['text'],
    num_rows: 19543
})
Val dataset: Dataset({
    features: ['text'],
    num_rows: 5190
})


In [39]:
# Measure base model preplexity on the ACTUAL val chunks
# We need this before we reload model for traning
val_sample = val_chunks[:50]
base_ppl_val = compute_preplixity(model, tokenizer, val_sample)
print(f"Base model preplexity on SEC validation chunks: {base_ppl_val:.1f}")
print("We'll compare this to the CPT model later")

Base model preplexity on SEC validation chunks: 15.4
We'll compare this to the CPT model later


### CPT with Unsloth + LoRA
1. Reload base model (fresh, no infrence mode)
2. Add LoRA adapters
3. Train on our SEC filing chunks

In [40]:
# Free memory from the infrence model
del model
torch.cuda.empty_cache()

# Reload fresh for traning
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

print(f"[OK] Fresh model loaded for training")


==((====))==  Unsloth 2025.11.1: Fast Llama patching. Transformers: 4.57.2.
   \\   /|    NVIDIA GeForce RTX 3070. Num GPUs = 1. Max memory: 8.0 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 8.6. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
[OK] Fresh model loaded for training


In [41]:
# Add LoRA adapters
#
# Key CPT Decisions:
# - rank=32: Higher than typical SFT (8-16) because we're teaching new knowledge, not just changing behaviour
# - target module includes embed_tokens + lm_head This is CPT-Specfic, SFT usually skips these. because we need the model to learn new domain vocablary
# - lora_alpha=32: Equal to rank, so effective scaling = 1

model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",   # Attention weights
        "gate_proj", "up_proj", "down_proj",      # FFN weights (where knowledge lives)
        "embed_tokens", "lm_head"                 # CPT-Specfic! (This is where model learns new vocab)
    ],
    lora_alpha=32,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    use_rslora=False, # false means scalling factor is lora_alpha/r which means 32/32=1 so effectie scale is 1
)


# How many parameters are we actually trainning?
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(f"\nTotal parameters:     {total_params:>12,}")
print(f"Trainable (LoRA):     {trainable_params:>12,}  ({100*trainable_params/total_params:.2f}%)")
print(f"Frozen (base model):  {frozen_params:>12,}  ({100*frozen_params/total_params:.2f}%)")

Unsloth: Offloading input_embeddings to disk to save VRAM
Unsloth: Offloading output_embeddings to disk to save VRAM


/home/junaid/miniconda3/envs/cuda13/lib/python3.11/site-packages/peft/tuners/tuners_utils.py:1348: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)


Unsloth: Training embed_tokens in mixed precision to save VRAM
Unsloth: Training lm_head in mixed precision to save VRAM

Total parameters:      176,134,464
Trainable (LoRA):       66,392,064  (37.69%)
Frozen (base model):   109,742,400  (62.31%)


In [42]:
from unsloth.trainer import UnslothTrainer, UnslothTrainingArguments

# Training configuration -- every parameter explained:
training_args = UnslothTrainingArguments(
    output_dir="./cpt_sec_training_args",

    # --- Core traning ---
    num_train_epochs=2,               # Cycles through entire dataset
    per_device_train_batch_size=32,   # 16 chunks per GPU step
    gradient_accumulation_steps=2,     # num of chunks traning sees before updating weights (effective batch size = 32 * 2 = 64)

    # --- Learning Rate ---
    learning_rate=2e-4,               # Standard for LoRA
    embedding_learning_rate=2e-5,     # 10x Lower for embeddings!
                                      # critical to avoid catestopic forgetting
    warmup_steps=50,                  # Gentely increase amount of weight updates
    lr_scheduler_type="cosine",       # gentler than a step-decay and helps the model converge smoothly at the end of training

    # --- Optmization ---
    optim="adamw_8bit",               # 8-biy AdamW saves VRAM
    weight_decay=0.01,                # L2 regularization
    max_grad_norm=0.3,                # Gradient clipping to prevent spikes

    # --- Data handling ---
    max_length=MAX_SEQ_LENGTH,
    packing=False,                    # avoid packer shape mismatches
    dataset_text_field="text",

    # --- Evaluation ---
    eval_strategy="steps",
    eval_steps=25,
    per_device_eval_batch_size=16,

    # --- Check pointing ---
    save_strategy="steps",
    save_steps=25,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",

    # --- logging ---
    logging_strategy="steps",
    logging_steps=10,
    seed=SEED,
)

trainer = UnslothTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

print("[OK] Trainer configured. Ready to train!")
print(f"Training samples: {len(train_dataset)}")
print(f"Eval samples: {len(val_dataset)}")

Unsloth: Tokenizing ["text"] (num_proc=20): 100%|██████████| 5190/5190 [00:03<00:00, 1638.98 examples/s]

[OK] Trainer configured. Ready to train!
Training samples: 19543
Eval samples: 5190


In [43]:
# TRAIN! (20-30 mins)

# Watch the eval_loss -- that's your north star.
# It should decrease and then plateau.

trainer_stats = trainer.train()
print(f"Final train loss: {trainer_stats.training_loss:.4f}")

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 2
   \\   /|    Num examples = 19,543 | Num Epochs = 2 | Total steps = 612
O^O/ \_/ \    Batch size per device = 32 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (32 x 2 x 1) = 64
 "-____-"     Trainable parameters = 66,392,064 of 229,218,624 (28.96% trained)


Step,Training Loss,Validation Loss
25,2.785500,2.695899
50,2.568900,2.544230
75,2.491500,2.457142
100,2.381200,2.407768
125,2.360800,2.373794
150,2.329800,2.348738
175,2.325900,2.328928
200,2.272000,2.314390
225,2.275400,2.302159
250,2.281500,2.290816


Final train loss: 2.2967


### Comparision Before vs After

In [44]:
# Switch to infrence mode
FastLanguageModel.for_inference(model)

# Generate from the same prompts we tried before
print("=" * 70)
print("AFTER CPT -- Financial Text Generation")
print("=" * 70)
for prompt in financial_prompts:
  completion = generate_text(model, tokenizer, prompt, max_new_tokens=60)
  print(f"\nPrompt: {prompt}")
  print(f"Output: {completion[:200]}")
  print("-" * 50)

AFTER CPT -- Financial Text Generation

Prompt: The company reported total revenue of
Output:  $152 million in 1984 and net income, loss for the years ended December 30 was approximately (or less) than expected. The Company's operating results have not been material as a result or related to t
--------------------------------------------------

Prompt: Risk factors include exposure to fluctuations in
Output:  the economy, environmental or other hazards. The Company is currently reviewing these and potential risks associated with its operations as it relates to natural disasters of various types (floods, e
--------------------------------------------------

Prompt: Diluted earnings per share for the fiscal year ended
Output:  December 31, 1987 were $.20/d and net income was .564% of weighted average common shares at December 31, 1987 compared to a gain of approximately $.25 in that same period (the "loss
--------------------------------------------------

Prompt: The Board of Directors 

In [45]:
# Preplexity comparision -- the real test
# Compare on the same val chunks we measured the base model on

cpt_ppl = compute_preplixity(model, tokenizer, val_sample)
print(f"\nPerplexity on SEC validation chunks:")
print(f"  Base model (before CPT):  {base_ppl_val:.1f}")
print(f"  After CPT:                {cpt_ppl:.1f}")
print(f"  Improvement:              {((base_ppl_val - cpt_ppl) / base_ppl_val * 100):.1f}%")
print()
if cpt_ppl < base_ppl_val:
  print("[PASS] Model is significantly less surprised by financial text!")
else:
  print("[WARN] Perplexity didn't improve -- might need more data or epochs")


Perplexity on SEC validation chunks:
  Base model (before CPT):  15.4
  After CPT:                9.0
  Improvement:              41.5%

[PASS] Model is significantly less surprised by financial text!


### The Forgetting Experiement

In [46]:
# Test on general English text
general_prompts = [
    "The cat sat on the",
    "In a galaxy far far away",
    "The best way to learn programming is to",
    "Once upon a time there was a",
    "The weather today is expected to be",
]

print("=" * 70)
print("AFTER CPT -- General English (Forgetting Check)")
print("=" * 70)
for prompt in general_prompts:
  completion = generate_text(model, tokenizer, prompt, max_new_tokens=60)
  print(f"\nPrompt: {prompt}")
  print(f"Output: {completion[:200]}")
  print("-" * 50)

AFTER CPT -- General English (Forgetting Check)

Prompt: The cat sat on the
Output:  table. The director of the company and an employee of the company were present at the meeting as well, which was held in the room where the meeting took place; however, it is not possible to determin
--------------------------------------------------

Prompt: In a galaxy far far away
Output: , the galaxy that we live in. We are at home with this computer and its software because of the fact that it is our most closely watched competitor to which all other competitors will be watching as w
--------------------------------------------------

Prompt: The best way to learn programming is to
Output:  do it. Programming involves many different areas of knowledge and skills, which may be taught in separate courses or through the use of specific software applications that have been developed by othe
--------------------------------------------------

Prompt: Once upon a time there was a
Output:  party which ha

In [47]:
# Perplexity on general text
general_texts = [
    "The weather was beautiful that morning as the children walked to school through the park.",
    "Scientists have discovered a new species of butterfly in the Amazon rainforest.",
    "The recipe calls for two cups of flour, one egg, and a tablespoon of butter.",
    "The basketball game went into overtime after a dramatic three-point shot.",
    "She opened the book and began reading the first chapter aloud to her students.",
]

general_ppl_after_cpt = compute_preplixity(model, tokenizer, general_texts)
print(f"Perplexity on general English (after CPT): {general_ppl_after_cpt:.1f}")

Perplexity on general English (after CPT): 34.7


- If this is much higher than expected, the model has 'forgotten'
- some general English. This is CATASTROPHIC FORGETTING in action.
- LoRA helps prevent this (base weights are frozen), but it's not perfect.
- Solution: mix in general data during training.

### CPT with Mixed Data(general english + SEC 10K filing)
80% domain data + 20% general data. This way model mantains its general capablites while learning domain knowledge

In [48]:
general_dataset = load_dataset("wikitext", "wikitext-103-raw-v1", split="train")
print(f"general_dataset: {general_dataset}")

# now we mix wiki data along with chunks
mixed_train_chunks = []
mixed_val_chunks = []

# --- Mix train chunks ---
for chunk in train_chunks:
    random_index = random.randrange(len(general_dataset))
    random_general_example = general_dataset[random_index]

    # Take 50 WORDS from the general example
    general_words = " ".join(random_general_example.get("text").split()[:50])

    mixed_train_chunks.append(chunk + " " + general_words)

# --- Mix val chunks ---
for chunk in val_chunks:
    random_index = random.randrange(len(general_dataset))
    random_general_example = general_dataset[random_index]

    # Take 50 WORDS from the general example
    general_words = " ".join(random_general_example.get("text").split()[:50])

    mixed_val_chunks.append(chunk + " " + general_words)

mixed_train_dataset = Dataset.from_dict({"text": mixed_train_chunks})
mixed_val_dataset = Dataset.from_dict({"text": mixed_val_chunks})

print(f"\nMixed_train_dataset: {mixed_train_dataset}")
print(f"\nMixed_val_dataset: {mixed_val_dataset}")

Generating validation split: 100%|██████████| 3760/3760 [00:00<00:00, 1056443.13 examples/s]


general_dataset: Dataset({
    features: ['text'],
    num_rows: 1801350
})

Mixed_train_dataset: Dataset({
    features: ['text'],
    num_rows: 19543
})

Mixed_val_dataset: Dataset({
    features: ['text'],
    num_rows: 5190
})


In [49]:
# Reload model for traning
del model, trainer
torch.cuda.empty_cache()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=[
        "k_proj", "q_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
        "embed_tokens", "lm_head",
    ],
    lora_alpha=32,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    use_rslora=False
)

print("[OK] Fresh model loaded for mixing experement")

==((====))==  Unsloth 2025.11.1: Fast Llama patching. Transformers: 4.57.2.
   \\   /|    NVIDIA GeForce RTX 3070. Num GPUs = 1. Max memory: 8.0 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 8.6. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Offloading input_embeddings to disk to save VRAM
Unsloth: Offloading output_embeddings to disk to save VRAM


/home/junaid/miniconda3/envs/cuda13/lib/python3.11/site-packages/peft/tuners/tuners_utils.py:1348: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)


Unsloth: Training embed_tokens in mixed precision to save VRAM
Unsloth: Training lm_head in mixed precision to save VRAM
[OK] Fresh model loaded for mixing experement


In [50]:
# Training configuration -- every parameter explained:
training_args_mix = UnslothTrainingArguments(
    output_dir="./cpt_sec_mixed_training_args",

    # --- Core traning ---
    num_train_epochs=2,               # Cycles through entire dataset
    per_device_train_batch_size=32,   # 32 chunks per GPU step
    gradient_accumulation_steps=2,    # num of chunks traning sees before updating weights (effective batch size = 32 * 2 = 64)

    # --- Learning Rate ---
    learning_rate=2e-4,               # Standard for LoRA
    embedding_learning_rate=2e-5,     # 10x Lower for embeddings!
                                      # critical to avoid catestopic forgetting
    warmup_steps=50,                  # Gentely increase amount of weight updates
    lr_scheduler_type="cosine",       # gentler than a step-decay and helps the model converge smoothly at the end of training

    # --- Optmization ---
    optim="adamw_8bit",               # 8-biy AdamW saves VRAM
    weight_decay=0.01,                # L2 regularization
    max_grad_norm=0.3,                # Gradient clipping to prevent spikes

    # --- Data handling ---
    max_length=MAX_SEQ_LENGTH,
    packing=False,                    # avoid packer shape mismatches
    dataset_text_field="text",

    # --- Evaluation ---
    eval_strategy="steps",
    eval_steps=25,
    per_device_eval_batch_size=16,

    # --- Check pointing ---
    save_strategy="steps",
    save_steps=25,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",

    # --- logging ---
    logging_strategy="steps",
    logging_steps=10,
    seed=SEED,
)

trainer_mix = UnslothTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args_mix,
    train_dataset=mixed_train_dataset,
    eval_dataset=mixed_val_dataset,
)

print("[OK] Trainer configured. Ready to train!")
print(f"Mixed Training samples: {len(mixed_train_dataset)}")
print(f"Mixed Eval samples: {len(mixed_val_dataset)}")

Unsloth: Tokenizing ["text"] (num_proc=20): 100%|██████████| 5190/5190 [00:03<00:00, 1460.93 examples/s]

[OK] Trainer configured. Ready to train!
Mixed Training samples: 19543
Mixed Eval samples: 5190


In [51]:
print("=" * 70)
print("TRAINING WITH DATA MIXING (80/20)")
print("=" * 70)

trainer_mix_stats = trainer_mix.train()
print(f"\nFinal train loss: {trainer_mix_stats.training_loss:.4f}")

The model is already on multiple devices. Skipping the move to device specified in `args`.


TRAINING WITH DATA MIXING (80/20)


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 2
   \\   /|    Num examples = 19,543 | Num Epochs = 2 | Total steps = 612
O^O/ \_/ \    Batch size per device = 32 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (32 x 2 x 1) = 64
 "-____-"     Trainable parameters = 66,392,064 of 229,218,624 (28.96% trained)


Step,Training Loss,Validation Loss
25,2.899000,2.816055
50,2.680200,2.661974
75,2.610100,2.569190
100,2.494500,2.515961
125,2.461500,2.479501
150,2.434500,2.452330
175,2.429500,2.431020
200,2.368900,2.415236
225,2.382100,2.402545
250,2.381600,2.391003



Final train loss: 2.4003


In [53]:
# Compare: Mixed vs Pure domain training
FastLanguageModel.for_inference(model)

mixed_sec_ppl = compute_preplixity(model, tokenizer, val_sample)
mixed_general_ppl = compute_preplixity(model, tokenizer, general_texts)

print("=" * 70)
print("FORGETTING EXPERIMENT RESULTS")
print("=" * 70)
print()
print(f"{'Metric':<35} {'Base':>10} {'Pure CPT':>10} {'Mixed CPT':>10}")
print("-" * 70)
print(f"{'Perplexity (SEC filings)':.<35} {base_ppl_val:>10.1f} {cpt_ppl:>10.1f} {mixed_sec_ppl:>10.1f}")
print(f"{'Perplexity (General English)':.<35} {'--':>10} {general_ppl_after_cpt:>10.1f} {mixed_general_ppl:>10.1f}")

RuntimeError: Expected all tensors to be on the same device, but got index is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA__index_select)

**What to look for:**
- Pure CPT: great on SEC, worse on general (forgetting!)
- Mixed CPT: slightly worse on SEC, but maintains general ability
- This is the tradeoff. In production, you almost always want mixing.

### Can it Answer Finance questions?

In [ ]:
print("=" * 70)
print("THE LIMIT OF CPT")
print("=" * 70)
print()

question_prompts = [
    "Question: What was Apple's total revenue in fiscal year 2023?\nAnswer:",
    "Question: What is EBITDA and why do investors care about it?\nAnswer:",
    "Question: Explain the difference between operating income and net income.\nAnswer:",
]

for prompt in question_prompts:
    completion = generate_text(model, tokenizer, prompt, max_new_tokens=80)
    print("-" * 50)
    print(f"Prompt: {prompt}")
    print(f"Output: {completion[:250]}")
    print("-" * 50)
    print()

Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


THE LIMIT OF CPT



Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--------------------------------------------------
Prompt: Question: What was Apple's total revenue in fiscal year 2023?
Answer:
Output:  Total revenues for the three years ended December 31, 2023 were $5.7 million (the following table includes changes from prior periods). The increase resulted primarily of lower sales and higher inventory levels due to a decrease in demand during thi
--------------------------------------------------



Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--------------------------------------------------
Prompt: Question: What is EBITDA and why do investors care about it?
Answer:
Output:  The most commonly accepted measure of a company's cash flows (and therefore, its net income) from operations has been the Company's Earnings Per Share. This metric measures earnings after subtracting interest expense on common stock dividends which 
--------------------------------------------------

--------------------------------------------------
Prompt: Question: Explain the difference between operating income and net income.
Answer:
Output:  Operating Income = (Cost of Goods Sold) - (Operating Expenses) = Net Losses on Sale 60% Of Cost To Date Average Inventory 52% Total Assets at December 31, 1987 = (Net Sales from Period Ending in December) = (Total Days Since Year Ended ) -----------
--------------------------------------------------



**Observation:**
- Model keeps generating text instead of answering the question.
- CPT taught it financial LANGUAGE, not the financial BEHAVIOR.

To make it actually answer questions, we need Supervised Fine Tuning (SFT)!

### Save Model

In [ ]:
# Save LoRA adapter
SAVE_PATH = "./cpt_sec_adapter"
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

adapter_size = sum(
    os.path.getsize(os.path.join(SAVE_PATH, f))
    for f in os.listdir(SAVE_PATH) if f.endswith(('.safetensors', '.bin'))
)

print(f"[OK] Adapter saved to {SAVE_PATH}")
print(f"Adapter size: {adapter_size / 1e6:.1f} MB")
print(f"Full model would be: ~270 MB")
print(f"Savings: {(1 - adapter_size / 270e6) * 100:.0f}%")

[OK] Adapter saved to ./cpt_sec_adapter
Adapter size: 215.4 MB
Full model would be: ~270 MB
Savings: 20%


---
## Summary

| Step | What | Why |
|------|------|-----|
| Base model test | Generated financial text with SmolLM | See it fail -- motivation for CPT |
| Data prep | Downloaded SEC 10-K filings, cleaned, chunked | Real-world data pipeline |
| CPT training | LoRA (rank 32) with Unsloth on domain text | Teach the model financial language |
| Before/after | Compared generation quality + perplexity | Prove it worked |
| Forgetting test | Checked general English after CPT | Understand the tradeoff |
| Data mixing | 80% domain + 20% general | Solution to catastrophic forgetting |
| SFT | Asked questions -- model can't answer | Next step after cpt |

**Key takeaway:** CPT teaches the model to SOUND like a domain. It does NOT teach the model to be HELPFUL. That's SFT.

**Next:** Planning to take this CPT'd model and fine-tune it on financial Q&A pairs (`virattt/financial-qa-10K`) so it actually answers questions about finance.